# 01. Быстрый старт: месячный nowcast ИПЦ

Ноутбук показывает минимальный цикл: фикстуры → признаки → бенчмарк → CatBoost → RMSE.

Данные — **фикстуры** `testdata/cpi_nowcast/` (2021–2022), а не живой ClickHouse. Числа здесь учебные и не являются каноническими: канонические считают скрипты `models/cpi/` и пинуют их тесты.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

ROOT = Path.cwd()
if not (ROOT / "testdata").exists():
    ROOT = ROOT.parent
ROOT

In [ ]:
ipc = pd.read_csv(ROOT / "testdata/cpi_nowcast/ipc_mes_2021_2022.csv", parse_dates=["date"])
usd = pd.read_csv(ROOT / "testdata/cpi_nowcast/usd_2021_2022.csv", parse_dates=["date"])
fao = pd.read_csv(ROOT / "testdata/cpi_nowcast/fao_2021_2022.csv", parse_dates=["date"])

ALL_CPI = "Индексы потребительских цен на товары и услуги по Российской Федерации"
cpi = ipc[ipc["name"] == ALL_CPI].copy()
cpi["y"] = cpi["percent"] - 100.0          # м/м, процентные пункты
cpi["month"] = cpi["date"].dt.to_period("M")

usd_m = usd.assign(month=usd["date"].dt.to_period("M")).groupby("month")["price"].mean()
fao_m = fao.assign(month=fao["date"].dt.to_period("M")).groupby("month")["price_index"].mean()

df = cpi.set_index("month")[["y"]].join(usd_m.rename("usd")).join(fao_m.rename("fao"))
df["y_lag1"] = df["y"].shift(1)
df["usd_chg"] = df["usd"].pct_change().shift(1) * 100
df["fao_chg"] = df["fao"].pct_change().shift(1) * 100
df = df.dropna()
df.head()

In [ ]:
train = df[df.index.year == 2021]
test = df[df.index.year == 2022]
features = ["y_lag1", "usd_chg", "fao_chg"]

model = CatBoostRegressor(
    iterations=200, depth=2, learning_rate=0.05,
    random_seed=42, verbose=False, allow_writing_files=False,
)
model.fit(train[features], train["y"])
pred_cb = model.predict(test[features])
pred_b1 = test["y_lag1"].to_numpy()        # B1: последнее значение

rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print(f"train rows={len(train)}, test rows={len(test)}")
print(f"B1 last value RMSE = {rmse(pred_b1 - test['y']):.3f} п.п.")
print(f"CatBoost       RMSE = {rmse(pred_cb - test['y']):.3f} п.п.")

## Что дальше

- Полный прогон с бенчмарками B1–B3 и walk-forward: `make cpi-nowcast ARGS="--quick"`.
- Как устроен walk-forward, purge/embargo и as-of — ноутбук `02_walk_forward_and_pit.ipynb`.
- Как сравнивать champion и challenger — `03_gate_dm_bootstrap.ipynb`.
